# 05 · Withheld-origin evaluation and controlled ablations

**Only here** may we open documented ignition labels. In this notebook, the reference is entirely SYNTHETIC and therefore proves pipeline operation only. For historical Camp Fire evaluation switch LABEL to `data/evaluation/camp_fire_reference.json` after collecting real data.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import json
import matplotlib.pyplot as plt
INPUT = ROOT / "data/synthetic/toy_firms.csv"  # SYNTHETIC DEMO, NOT historical NASA data
WEATHER = ROOT / "data/synthetic/toy_weather.json"
assert INPUT.exists(), "Run `python scripts/make_synthetic.py` from repository root first"
print("Data mode: SYNTHETIC - demonstration only")

In [ ]:
from wildfire.ingest import load_firms,load_weather
from wildfire.inference import infer_origin,InferenceOptions
from wildfire.evaluation import evaluate_result,ablation_study
sat=load_firms(INPUT);wx=load_weather(WEATHER)
options=InferenceOptions(cell_m=850,margin_km=9,ensemble=True)
# Model is fitted FIRST; label is loaded only to score results.
model=infer_origin(sat,wx,options,variant='fusion')
LABEL=ROOT/'data/synthetic/toy_reference.json'
metrics=evaluate_result(model,str(LABEL))
print(json.dumps(metrics,indent=2))

In [ ]:
# Ablations have matched candidate boundaries and parameters.
rows=ablation_study(sat,wx,str(LABEL),options=options)
for r in rows:print(r)
fig,ax=plt.subplots(figsize=(7,4))
ax.bar([r['variant'] for r in rows],[r['model_error_km'] for r in rows])
ax.axhline(rows[0]['centroid_baseline_error_km'],linestyle='--',color='black',label='Earliest centroid')
ax.set(ylabel='Location error (km)',title='Synthetic ablation only — NOT real wildfire results')
ax.legend();plt.show()

In [ ]:
# The reference file is never input to candidate selection or physical graph edge costs.
summary={'reference_type':'SYNTHETIC - demonstration only','evaluation':metrics,'ablations':rows}
out=ROOT/'artifacts/05_evaluation.json';out.write_text(json.dumps(summary,indent=2))
print('Saved',out)

**Controls:** Do not choose an observation cutoff, search margin, propagation speed or favourable model based on the documented ignition. A single historical case is a case study; repeat on independent fires for any generalisation claim.